In [2]:
# -*- coding: utf-8 -*-
"""
循环构建不同缓冲距离下的随机森林模型，并计算 OOB R² 与 OOB RMSE。

因变量：
lst0603, lst0806, lst0828, lst0803, lst0619, lst0719

缓冲距离：
30–300 m，步长 30 m

固定变量：
TCC, GCI, BCI, WCI, RCI, Shape_Area,
jungong, BAH, BHSD, DIST

缓冲区变量：
TCC_bXm, GCI_bXm, BCI_bXm, WCI_bXm,
RCI_bXm, BAH_bXm, BHSD_bXm

说明：
1. 因变量中的 0 按缺失值处理并删除；
2. 自变量缺失值用当前模型样本内的中位数填补；
3. 采用随机森林袋外预测（OOB prediction）计算 R² 和 RMSE；
4. 每个缓冲距离均重新构建模型；
5. 输出结果表、一个六场景综合双 y 轴图，以及六张单场景双 y 轴图。

首次运行若缺少依赖，可在 Jupyter Notebook 中执行：

!pip install pandas numpy scikit-learn matplotlib openpyxl xlrd
"""

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score


warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)


# =============================================================================
# 0. 参数设置
# =============================================================================

DATA_PATH = Path(
    r"E:\excel\FILES\博士申请\第二篇论文\data"
    r"\wuhuanshiliang\SH_attributes.xls"
)

OUTPUT_DIR = Path(
    r"E:\excel\FILES\博士申请\第二篇论文"
    r"\脚本\Python"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# 因变量
TARGETS = [
    "lst0603",
    "lst0806",
    "lst0828",
    "lst0803",
    "lst0619",
    "lst0719",
]


# 30–300 m，每 30 m 一档
BUFFER_DISTANCES = list(
    range(30, 301, 30)
)


# 固定变量
FIXED_VARS = [
    "TCC",
    "GCI",
    "BCI",
    "WCI",
    "RCI",
    "Shape_Area",
    "jungong",
    "BAH",
    "BHSD",
    "DIST",
]


# 缓冲区变量的基础名称
BUFFER_BASE_VARS = [
    "TCC",
    "GCI",
    "BCI",
    "WCI",
    "RCI",
    "BAH",
    "BHSD",
]


# =============================================================================
# 随机森林参数
# =============================================================================

N_TREES = 1000

MIN_NODE_SIZE = 5

RANDOM_STATE = 42

N_JOBS = -1


# sklearn 回归随机森林默认 max_features=1.0，
# 即每次分裂考虑全部变量。
#
# 若希望采用更强的随机特征抽样，可改为：
# MAX_FEATURES = "sqrt"
#
# 或：
# MAX_FEATURES = 1 / 3

MAX_FEATURES = 1.0


# =============================================================================
# 场景分组和绘图参数
# =============================================================================

ORANGE_TARGETS = [
    "lst0603",
    "lst0806",
    "lst0828",
]

RED_TARGETS = [
    "lst0803",
    "lst0619",
    "lst0719",
]


GROUP_COLORS = {
    "orange": "#E69F00",
    "red": "#EA1818",
}


TARGET_MARKERS = {
    "lst0603": "o",
    "lst0806": "s",
    "lst0828": "^",
    "lst0803": "o",
    "lst0619": "s",
    "lst0719": "^",
}


# =============================================================================
# 字体与坐标轴参数
# =============================================================================

AXIS_LABEL_SIZE = 14

TICK_LABEL_SIZE = 14

TITLE_SIZE_SINGLE = 13

TITLE_SIZE_ALL = 14

LEGEND_SIZE_SINGLE = 11

LEGEND_SIZE_ALL = 10


# =============================================================================
# 1. 辅助函数
# =============================================================================

def get_buffer_vars(
    distance: int,
) -> list[str]:
    """
    生成指定缓冲距离对应的变量名。

    例如：
    distance=90

    返回：
    [
        "TCC_b90m",
        "GCI_b90m",
        ...
    ]
    """

    return [
        f"{var}_b{distance}m"
        for var in BUFFER_BASE_VARS
    ]


def validate_columns(
    df: pd.DataFrame,
) -> None:
    """
    一次性检查脚本所需字段是否全部存在。
    """

    required = set(
        TARGETS + FIXED_VARS
    )

    for distance in BUFFER_DISTANCES:
        required.update(
            get_buffer_vars(distance)
        )

    missing = sorted(
        required.difference(df.columns)
    )

    if missing:
        raise KeyError(
            "数据中缺少以下必要字段：\n"
            + "\n".join(missing)
        )


def fit_one_rf(
    df: pd.DataFrame,
    target: str,
    distance: int,
) -> dict:
    """
    针对一个因变量和一个缓冲距离构建随机森林模型，
    并返回 OOB R² 和 OOB RMSE。
    """

    predictors = (
        FIXED_VARS
        + get_buffer_vars(distance)
    )

    model_df = df[
        [target] + predictors
    ].copy()


    # -------------------------------------------------------------------------
    # 将所有字段转为数值
    # -------------------------------------------------------------------------

    model_df = model_df.apply(
        pd.to_numeric,
        errors="coerce",
    )

    model_df.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True,
    )


    # -------------------------------------------------------------------------
    # 将因变量中的 0 处理为缺失值
    # -------------------------------------------------------------------------

    model_df.loc[
        model_df[target] == 0,
        target,
    ] = np.nan


    # 只删除因变量缺失的样本
    model_df = (
        model_df
        .dropna(subset=[target])
        .reset_index(drop=True)
    )


    if len(model_df) < 30:
        raise ValueError(
            f"{target} 在 {distance} m 模型中"
            f"仅剩 {len(model_df)} 个有效样本，"
            f"无法稳定建模。"
        )


    # -------------------------------------------------------------------------
    # 提取自变量和因变量
    # -------------------------------------------------------------------------

    X = model_df[predictors]

    y = model_df[target].to_numpy(
        dtype=float
    )


    # -------------------------------------------------------------------------
    # 检查是否有整列缺失
    # -------------------------------------------------------------------------

    all_missing_cols = (
        X.columns[
            X.isna().all()
        ]
        .tolist()
    )

    if all_missing_cols:
        raise ValueError(
            f"{target}–{distance} m 模型"
            f"存在全为空的自变量："
            f"{all_missing_cols}"
        )


    # -------------------------------------------------------------------------
    # 中位数填补自变量缺失值
    # -------------------------------------------------------------------------

    imputer = SimpleImputer(
        strategy="median"
    )

    X_imputed = imputer.fit_transform(X)


    # -------------------------------------------------------------------------
    # 构建随机森林
    # -------------------------------------------------------------------------

    rf = RandomForestRegressor(
        n_estimators=N_TREES,
        min_samples_leaf=MIN_NODE_SIZE,
        max_features=MAX_FEATURES,
        bootstrap=True,
        oob_score=True,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )

    rf.fit(
        X_imputed,
        y,
    )


    # -------------------------------------------------------------------------
    # 提取 OOB 预测
    # -------------------------------------------------------------------------

    oob_pred = rf.oob_prediction_

    valid_oob = np.isfinite(
        oob_pred
    )

    if valid_oob.sum() == 0:
        raise RuntimeError(
            f"{target}–{distance} m 模型"
            f"未生成有效 OOB 预测。"
        )


    # -------------------------------------------------------------------------
    # 计算 OOB R² 和 OOB RMSE
    # -------------------------------------------------------------------------

    r2 = r2_score(
        y[valid_oob],
        oob_pred[valid_oob],
    )

    rmse = np.sqrt(
        mean_squared_error(
            y[valid_oob],
            oob_pred[valid_oob],
        )
    )


    return {
        "target": target,
        "buffer_distance_m": distance,
        "n_samples": int(len(y)),
        "n_predictors": int(len(predictors)),
        "r2_oob": float(r2),
        "rmse_oob": float(rmse),
        "n_trees": N_TREES,
        "min_node_size": MIN_NODE_SIZE,
        "max_features": str(MAX_FEATURES),
    }


def style_axes(
    ax1: plt.Axes,
    ax2=None,
) -> None:
    """
    统一设置：

    1. 横轴格式；
    2. 左侧纵轴格式；
    3. 右侧纵轴格式；
    4. 坐标轴刻度数字字号。
    """

    # -------------------------------------------------------------------------
    # 左侧坐标轴和横轴
    # -------------------------------------------------------------------------

    ax1.spines["top"].set_visible(
        False
    )

    ax1.tick_params(
        axis="both",
        direction="out",
        width=1.1,
        labelsize=TICK_LABEL_SIZE,
    )

    ax1.set_xticks(
        BUFFER_DISTANCES
    )


    # -------------------------------------------------------------------------
    # 右侧纵轴
    # -------------------------------------------------------------------------

    if ax2 is not None:

        ax2.spines["top"].set_visible(
            False
        )

        ax2.tick_params(
            axis="y",
            direction="out",
            width=1.1,
            labelsize=TICK_LABEL_SIZE,
        )


def get_target_color(
    target: str,
) -> str:
    """
    根据目标变量所属分组返回绘图颜色。
    """

    if target in ORANGE_TARGETS:
        return GROUP_COLORS["orange"]

    return GROUP_COLORS["red"]


# =============================================================================
# 2. 单场景绘图
# =============================================================================

def plot_single_target(
    results: pd.DataFrame,
    target: str,
) -> None:
    """
    为单个 LST 场景绘制 R² 与 RMSE 双 y 轴折线图。
    """

    sub = (
        results.loc[
            results["target"] == target
        ]
        .sort_values(
            "buffer_distance_m"
        )
        .copy()
    )


    color = get_target_color(target)

    marker = TARGET_MARKERS[target]


    # -------------------------------------------------------------------------
    # 创建双 y 轴图
    # -------------------------------------------------------------------------

    fig, ax1 = plt.subplots(
        figsize=(8.6, 5.6),
        dpi=160,
    )

    ax2 = ax1.twinx()


    # -------------------------------------------------------------------------
    # 左轴：OOB R²
    # -------------------------------------------------------------------------

    line_r2 = ax1.plot(
        sub["buffer_distance_m"],
        sub["r2_oob"],
        color=color,
        linestyle="-",
        marker=marker,
        linewidth=2.2,
        markersize=6,
        label=f"{target} R²",
    )[0]


    # -------------------------------------------------------------------------
    # 右轴：OOB RMSE
    # -------------------------------------------------------------------------

    line_rmse = ax2.plot(
        sub["buffer_distance_m"],
        sub["rmse_oob"],
        color=color,
        linestyle="--",
        marker=marker,
        linewidth=2.2,
        markersize=6,
        markerfacecolor="white",
        label=f"{target} RMSE",
    )[0]


    # -------------------------------------------------------------------------
    # 坐标轴标题和图标题
    # -------------------------------------------------------------------------

    ax1.set_xlabel(
        "Buffer distance (m)",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax1.set_ylabel(
        "OOB R²",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax2.set_ylabel(
        "OOB RMSE",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax1.set_title(
        f"RF performance across buffer distances: {target}",
        fontsize=TITLE_SIZE_SINGLE,
    )


    # -------------------------------------------------------------------------
    # 统一设置左右坐标轴
    # -------------------------------------------------------------------------

    style_axes(
        ax1,
        ax2,
    )


    # -------------------------------------------------------------------------
    # 图例
    # -------------------------------------------------------------------------

    ax1.legend(
        [
            line_r2,
            line_rmse,
        ],
        [
            line_r2.get_label(),
            line_rmse.get_label(),
        ],
        loc="best",
        frameon=False,
        fontsize=LEGEND_SIZE_SINGLE,
    )


    # -------------------------------------------------------------------------
    # 保存图片
    # -------------------------------------------------------------------------

    fig.tight_layout()

    fig.savefig(
        OUTPUT_DIR
        / f"RF_{target}_R2_RMSE_dual_axis.png",
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        OUTPUT_DIR
        / f"RF_{target}_R2_RMSE_dual_axis.pdf",
        bbox_inches="tight",
    )

    plt.close(fig)


# =============================================================================
# 3. 六场景综合绘图
# =============================================================================

def plot_all_targets(
    results: pd.DataFrame,
) -> None:
    """
    将六个 LST 场景的 R² 与 RMSE
    绘制在同一张双 y 轴图中。
    """

    fig, ax1 = plt.subplots(
        figsize=(12.5, 7.2),
        dpi=160,
    )

    ax2 = ax1.twinx()


    legend_handles = []

    legend_labels = []


    # -------------------------------------------------------------------------
    # 循环绘制六个场景
    # -------------------------------------------------------------------------

    for target in TARGETS:

        sub = (
            results.loc[
                results["target"] == target
            ]
            .sort_values(
                "buffer_distance_m"
            )
            .copy()
        )


        color = get_target_color(target)

        marker = TARGET_MARKERS[target]


        # 左轴：R²
        line_r2 = ax1.plot(
            sub["buffer_distance_m"],
            sub["r2_oob"],
            color=color,
            linestyle="-",
            marker=marker,
            linewidth=2.0,
            markersize=5.5,
            alpha=0.95,
            label=f"{target} R²",
        )[0]


        # 右轴：RMSE
        line_rmse = ax2.plot(
            sub["buffer_distance_m"],
            sub["rmse_oob"],
            color=color,
            linestyle="--",
            marker=marker,
            linewidth=2.0,
            markersize=5.5,
            markerfacecolor="white",
            alpha=0.95,
            label=f"{target} RMSE",
        )[0]


        legend_handles.extend(
            [
                line_r2,
                line_rmse,
            ]
        )

        legend_labels.extend(
            [
                line_r2.get_label(),
                line_rmse.get_label(),
            ]
        )


    # -------------------------------------------------------------------------
    # 坐标轴标题和图标题
    # -------------------------------------------------------------------------

    ax1.set_xlabel(
        "Buffer distance (m)",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax1.set_ylabel(
        "OOB R²",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax2.set_ylabel(
        "OOB RMSE",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax1.set_title(
        "Random forest performance across buffer distances",
        fontsize=TITLE_SIZE_ALL,
    )


    # -------------------------------------------------------------------------
    # 统一设置左轴、横轴和右轴
    # -------------------------------------------------------------------------

    style_axes(
        ax1,
        ax2,
    )


    # -------------------------------------------------------------------------
    # 图例
    # -------------------------------------------------------------------------

    ax1.legend(
        legend_handles,
        legend_labels,
        ncol=3,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.13),
        frameon=False,
        fontsize=LEGEND_SIZE_ALL,
    )


    # -------------------------------------------------------------------------
    # 保存图片
    # -------------------------------------------------------------------------

    fig.tight_layout(
        rect=[0, 0.08, 1, 1]
    )

    fig.savefig(
        OUTPUT_DIR
        / "RF_all_LST_R2_RMSE_dual_axis.png",
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        OUTPUT_DIR
        / "RF_all_LST_R2_RMSE_dual_axis.pdf",
        bbox_inches="tight",
    )

    plt.close(fig)


# =============================================================================
# 4. 主程序
# =============================================================================

def main() -> None:
    """
    主程序：
    读取数据、循环建模、输出结果并绘图。
    """

    print(
        f"正在读取数据：{DATA_PATH}"
    )


    # -------------------------------------------------------------------------
    # 读取数据
    # -------------------------------------------------------------------------

    df = pd.read_excel(
        DATA_PATH,
        sheet_name=0,
        engine="xlrd",
    )


    print(
        f"数据维度："
        f"{df.shape[0]} 行 × "
        f"{df.shape[1]} 列"
    )


    # -------------------------------------------------------------------------
    # 检查字段
    # -------------------------------------------------------------------------

    validate_columns(df)

    print(
        "字段检查通过。"
    )


    # -------------------------------------------------------------------------
    # 循环构建模型
    # -------------------------------------------------------------------------

    all_results = []

    total_models = (
        len(TARGETS)
        * len(BUFFER_DISTANCES)
    )

    model_no = 0


    for target in TARGETS:

        for distance in BUFFER_DISTANCES:

            model_no += 1


            print(
                f"[{model_no:02d}/{total_models}] "
                f"建模："
                f"target={target}, "
                f"buffer={distance} m"
            )


            result = fit_one_rf(
                df=df,
                target=target,
                distance=distance,
            )


            all_results.append(
                result
            )


            print(
                f"    n={result['n_samples']}, "
                f"OOB R²={result['r2_oob']:.4f}, "
                f"OOB RMSE={result['rmse_oob']:.4f}"
            )


    # -------------------------------------------------------------------------
    # 整理结果
    # -------------------------------------------------------------------------

    results = pd.DataFrame(
        all_results
    )

    results = (
        results
        .sort_values(
            [
                "target",
                "buffer_distance_m",
            ]
        )
        .reset_index(drop=True)
    )


    # -------------------------------------------------------------------------
    # 按最高 R² 提取最优距离
    # -------------------------------------------------------------------------

    best_by_r2 = (
        results.loc[
            results
            .groupby("target")["r2_oob"]
            .idxmax()
        ]
        .sort_values("target")
        .reset_index(drop=True)
    )


    # -------------------------------------------------------------------------
    # 按最低 RMSE 提取最优距离
    # -------------------------------------------------------------------------

    best_by_rmse = (
        results.loc[
            results
            .groupby("target")["rmse_oob"]
            .idxmin()
        ]
        .sort_values("target")
        .reset_index(drop=True)
    )


    # -------------------------------------------------------------------------
    # 输出路径
    # -------------------------------------------------------------------------

    csv_path = (
        OUTPUT_DIR
        / "RF_buffer_distance_metrics.csv"
    )

    xlsx_path = (
        OUTPUT_DIR
        / "RF_buffer_distance_metrics.xlsx"
    )


    # -------------------------------------------------------------------------
    # 保存 CSV
    # -------------------------------------------------------------------------

    results.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig",
    )


    # -------------------------------------------------------------------------
    # 保存 Excel
    # -------------------------------------------------------------------------

    with pd.ExcelWriter(
        xlsx_path,
        engine="openpyxl",
    ) as writer:

        results.to_excel(
            writer,
            sheet_name="all_metrics",
            index=False,
        )

        best_by_r2.to_excel(
            writer,
            sheet_name="best_by_R2",
            index=False,
        )

        best_by_rmse.to_excel(
            writer,
            sheet_name="best_by_RMSE",
            index=False,
        )


    # -------------------------------------------------------------------------
    # 绘制综合图
    # -------------------------------------------------------------------------

    plot_all_targets(
        results
    )


    # -------------------------------------------------------------------------
    # 绘制六张单场景图
    # -------------------------------------------------------------------------

    for target in TARGETS:

        plot_single_target(
            results,
            target,
        )


    # -------------------------------------------------------------------------
    # 输出运行结果
    # -------------------------------------------------------------------------

    print(
        "\n全部模型计算完成。"
    )

    print(
        f"结果表：{xlsx_path}"
    )

    print(
        "综合图："
        f"{OUTPUT_DIR / 'RF_all_LST_R2_RMSE_dual_axis.png'}"
    )

    print(
        "\n各场景最高 OOB R² 对应距离："
    )

    print(
        best_by_r2[
            [
                "target",
                "buffer_distance_m",
                "r2_oob",
                "rmse_oob",
            ]
        ].to_string(
            index=False
        )
    )


# =============================================================================
# 5. 运行主程序
# =============================================================================

if __name__ == "__main__":
    main()

正在读取数据：E:\excel\FILES\博士申请\第二篇论文\data\wuhuanshiliang\SH_attributes.xls
数据维度：1337 行 × 166 列
字段检查通过。
[01/60] 建模：target=lst0603, buffer=30 m
    n=1337, OOB R²=0.5054, OOB RMSE=1.0669
[02/60] 建模：target=lst0603, buffer=60 m
    n=1337, OOB R²=0.5302, OOB RMSE=1.0398
[03/60] 建模：target=lst0603, buffer=90 m
    n=1337, OOB R²=0.5415, OOB RMSE=1.0273
[04/60] 建模：target=lst0603, buffer=120 m
    n=1337, OOB R²=0.5336, OOB RMSE=1.0361
[05/60] 建模：target=lst0603, buffer=150 m
    n=1337, OOB R²=0.5255, OOB RMSE=1.0450
[06/60] 建模：target=lst0603, buffer=180 m
    n=1337, OOB R²=0.5252, OOB RMSE=1.0453
[07/60] 建模：target=lst0603, buffer=210 m
    n=1337, OOB R²=0.5192, OOB RMSE=1.0519
[08/60] 建模：target=lst0603, buffer=240 m
    n=1337, OOB R²=0.5169, OOB RMSE=1.0544
[09/60] 建模：target=lst0603, buffer=270 m
    n=1337, OOB R²=0.5037, OOB RMSE=1.0688
[10/60] 建模：target=lst0603, buffer=300 m
    n=1337, OOB R²=0.4955, OOB RMSE=1.0776
[11/60] 建模：target=lst0806, buffer=30 m
    n=1337, OOB R²=0.5815, OOB RMSE

In [1]:
# -*- coding: utf-8 -*-
"""
循环构建基线模型及不同缓冲距离下的随机森林模型，
并计算 OOB R² 与 OOB RMSE。

因变量
------
lst0603, lst0806, lst0828, lst0803, lst0619, lst0719

模型设置
--------
1. 基线模型（buffer distance = 0 m）：
   仅使用内部变量与控制变量，不加入任何缓冲区变量。

2. 缓冲区模型（buffer distance = 30–300 m，步长30 m）：
   使用内部变量、控制变量，以及相应距离的缓冲区变量。

固定变量（内部变量与控制变量）
----------------------------
TCC, GCI, BCI, WCI, RCI, Shape_Area,
jungong, BAH, BHSD, DIST

缓冲区变量
----------
TCC_bXm, GCI_bXm, BCI_bXm, WCI_bXm,
RCI_bXm, BAH_bXm, BHSD_bXm

说明
----
1. 因变量中的0按缺失值处理并删除；
2. 自变量缺失值用当前模型样本内的中位数填补；
3. 采用随机森林袋外预测（OOB prediction）计算R²和RMSE；
4. 基线模型以及每个缓冲距离模型均重新独立构建；
5. 输出全部模型结果、最佳模型结果、最佳缓冲区模型结果；
6. 输出一个六场景综合双y轴图，以及六张单场景双y轴图；
7. 图中buffer distance = 0表示不包含缓冲区变量的基线模型。

首次运行若缺少依赖，可在Jupyter Notebook中执行：

!pip install pandas numpy scikit-learn matplotlib openpyxl xlrd
"""

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score


warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)


# =============================================================================
# 0. 参数设置
# =============================================================================

DATA_PATH = Path(
    r"E:\excel\FILES\博士申请\第二篇论文\data"
    r"\wuhuanshiliang\SH_attributes.xls"
)

OUTPUT_DIR = Path(
    r"E:\excel\FILES\博士申请\第二篇论文"
    r"\脚本\Python"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# 因变量
TARGETS = [
    "lst0603",
    "lst0806",
    "lst0828",
    "lst0803",
    "lst0619",
    "lst0719",
]


# 实际缓冲距离：30–300 m，每30 m一档
BUFFER_DISTANCES = list(
    range(30, 301, 30)
)


# 全部模型距离：
# 0 m表示不加入任何缓冲区变量的基线模型
MODEL_DISTANCES = [
    0,
    *BUFFER_DISTANCES,
]


# 固定变量：
# 内部变量 + 控制变量
FIXED_VARS = [
    "TCC",
    "GCI",
    "BCI",
    "WCI",
    "RCI",
    "Shape_Area",
    "jungong",
    "BAH",
    "BHSD",
    "DIST",
]


# 缓冲区变量的基础名称
BUFFER_BASE_VARS = [
    "TCC",
    "GCI",
    "BCI",
    "WCI",
    "RCI",
    "BAH",
    "BHSD",
]


# =============================================================================
# 随机森林参数
# =============================================================================

N_TREES = 1000

MIN_NODE_SIZE = 5

RANDOM_STATE = 42

N_JOBS = -1


# sklearn回归随机森林默认max_features=1.0，
# 即每次分裂考虑全部变量。
#
# 若希望采用更强的随机特征抽样，可改为：
# MAX_FEATURES = "sqrt"
#
# 或：
# MAX_FEATURES = 1 / 3

MAX_FEATURES = 1.0


# =============================================================================
# 场景分组和绘图参数
# =============================================================================

ORANGE_TARGETS = [
    "lst0603",
    "lst0806",
    "lst0828",
]

RED_TARGETS = [
    "lst0803",
    "lst0619",
    "lst0719",
]


GROUP_COLORS = {
    "orange": "#E69F00",
    "red": "#EA1818",
}


TARGET_MARKERS = {
    "lst0603": "o",
    "lst0806": "s",
    "lst0828": "^",
    "lst0803": "o",
    "lst0619": "s",
    "lst0719": "^",
}


# =============================================================================
# 字体与坐标轴参数
# =============================================================================

AXIS_LABEL_SIZE = 14

TICK_LABEL_SIZE = 14

TITLE_SIZE_SINGLE = 13

TITLE_SIZE_ALL = 14

LEGEND_SIZE_SINGLE = 11

LEGEND_SIZE_ALL = 10


# =============================================================================
# 1. 辅助函数
# =============================================================================

def get_buffer_vars(
    distance: int,
) -> list[str]:
    """
    生成指定距离对应的缓冲区变量名。

    当distance=0时，返回空列表，表示基线模型
    不包含任何缓冲区变量。

    例如
    ----
    distance=0：
        []

    distance=90：
        [
            "TCC_b90m",
            "GCI_b90m",
            "BCI_b90m",
            "WCI_b90m",
            "RCI_b90m",
            "BAH_b90m",
            "BHSD_b90m"
        ]
    """

    if distance == 0:
        return []

    if distance not in BUFFER_DISTANCES:
        raise ValueError(
            f"不支持的缓冲距离：{distance} m。"
            f"允许的距离为0或{BUFFER_DISTANCES}。"
        )

    return [
        f"{var}_b{distance}m"
        for var in BUFFER_BASE_VARS
    ]


def get_model_type(
    distance: int,
) -> str:
    """
    根据距离返回模型类型。

    0 m：
        Baseline

    30–300 m：
        Buffer
    """

    if distance == 0:
        return "Baseline"

    return "Buffer"


def validate_columns(
    df: pd.DataFrame,
) -> None:
    """
    一次性检查脚本所需字段是否全部存在。

    基线模型只检查TARGETS和FIXED_VARS；
    缓冲区模型检查30–300 m对应的缓冲区变量。
    """

    required = set(
        TARGETS + FIXED_VARS
    )

    # 这里只循环真实缓冲距离，不检查不存在的_b0m字段
    for distance in BUFFER_DISTANCES:
        required.update(
            get_buffer_vars(distance)
        )

    missing = sorted(
        required.difference(df.columns)
    )

    if missing:
        raise KeyError(
            "数据中缺少以下必要字段：\n"
            + "\n".join(missing)
        )


def fit_one_rf(
    df: pd.DataFrame,
    target: str,
    distance: int,
) -> dict:
    """
    针对一个因变量和一个模型距离构建随机森林模型，
    并返回OOB R²和OOB RMSE。

    distance=0：
        仅使用FIXED_VARS，作为基线模型。

    distance>0：
        使用FIXED_VARS和相应距离的缓冲区变量。
    """

    buffer_vars = get_buffer_vars(
        distance
    )

    predictors = (
        FIXED_VARS
        + buffer_vars
    )

    model_type = get_model_type(
        distance
    )

    model_df = df[
        [target] + predictors
    ].copy()


    # -------------------------------------------------------------------------
    # 将所有字段转为数值
    # -------------------------------------------------------------------------

    model_df = model_df.apply(
        pd.to_numeric,
        errors="coerce",
    )

    model_df.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True,
    )


    # -------------------------------------------------------------------------
    # 将因变量中的0处理为缺失值
    # -------------------------------------------------------------------------

    model_df.loc[
        model_df[target] == 0,
        target,
    ] = np.nan


    # 只删除因变量缺失的样本
    model_df = (
        model_df
        .dropna(subset=[target])
        .reset_index(drop=True)
    )


    if len(model_df) < 30:
        raise ValueError(
            f"{target}在{distance} m模型中"
            f"仅剩{len(model_df)}个有效样本，"
            f"无法稳定建模。"
        )


    # -------------------------------------------------------------------------
    # 提取自变量和因变量
    # -------------------------------------------------------------------------

    X = model_df[
        predictors
    ]

    y = model_df[
        target
    ].to_numpy(
        dtype=float
    )


    # -------------------------------------------------------------------------
    # 检查是否存在整列缺失
    # -------------------------------------------------------------------------

    all_missing_cols = (
        X.columns[
            X.isna().all()
        ]
        .tolist()
    )

    if all_missing_cols:
        raise ValueError(
            f"{target}–{distance} m模型"
            f"存在全为空的自变量："
            f"{all_missing_cols}"
        )


    # -------------------------------------------------------------------------
    # 中位数填补自变量缺失值
    # -------------------------------------------------------------------------

    imputer = SimpleImputer(
        strategy="median"
    )

    X_imputed = imputer.fit_transform(
        X
    )


    # -------------------------------------------------------------------------
    # 构建随机森林
    # -------------------------------------------------------------------------

    rf = RandomForestRegressor(
        n_estimators=N_TREES,
        min_samples_leaf=MIN_NODE_SIZE,
        max_features=MAX_FEATURES,
        bootstrap=True,
        oob_score=True,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )

    rf.fit(
        X_imputed,
        y,
    )


    # -------------------------------------------------------------------------
    # 提取OOB预测
    # -------------------------------------------------------------------------

    oob_pred = rf.oob_prediction_

    valid_oob = np.isfinite(
        oob_pred
    )

    if valid_oob.sum() == 0:
        raise RuntimeError(
            f"{target}–{distance} m模型"
            f"未生成有效OOB预测。"
        )


    # -------------------------------------------------------------------------
    # 计算OOB R²和OOB RMSE
    # -------------------------------------------------------------------------

    r2 = r2_score(
        y[valid_oob],
        oob_pred[valid_oob],
    )

    rmse = np.sqrt(
        mean_squared_error(
            y[valid_oob],
            oob_pred[valid_oob],
        )
    )


    return {
        "target": target,
        "model_type": model_type,
        "buffer_distance_m": distance,
        "n_samples": int(len(y)),
        "n_predictors": int(len(predictors)),
        "n_internal_control_vars": int(
            len(FIXED_VARS)
        ),
        "n_buffer_vars": int(
            len(buffer_vars)
        ),
        "r2_oob": float(r2),
        "rmse_oob": float(rmse),
        "n_trees": N_TREES,
        "min_node_size": MIN_NODE_SIZE,
        "max_features": str(MAX_FEATURES),
    }


def style_axes(
    ax1: plt.Axes,
    ax2=None,
) -> None:
    """
    统一设置横轴、左侧纵轴和右侧纵轴格式。

    横轴中的0表示基线模型。
    """

    # -------------------------------------------------------------------------
    # 左侧坐标轴和横轴
    # -------------------------------------------------------------------------

    ax1.spines["top"].set_visible(
        False
    )

    ax1.tick_params(
        axis="both",
        direction="out",
        width=1.1,
        labelsize=TICK_LABEL_SIZE,
    )

    # 包括0 m基线模型
    ax1.set_xticks(
        MODEL_DISTANCES
    )

    ax1.set_xlim(
        min(MODEL_DISTANCES) - 8,
        max(MODEL_DISTANCES) + 8,
    )


    # -------------------------------------------------------------------------
    # 右侧纵轴
    # -------------------------------------------------------------------------

    if ax2 is not None:

        ax2.spines["top"].set_visible(
            False
        )

        ax2.tick_params(
            axis="y",
            direction="out",
            width=1.1,
            labelsize=TICK_LABEL_SIZE,
        )


def get_target_color(
    target: str,
) -> str:
    """
    根据目标变量所属分组返回绘图颜色。
    """

    if target in ORANGE_TARGETS:
        return GROUP_COLORS["orange"]

    return GROUP_COLORS["red"]


# =============================================================================
# 2. 单场景绘图
# =============================================================================

def plot_single_target(
    results: pd.DataFrame,
    target: str,
) -> None:
    """
    为单个LST场景绘制R²与RMSE双y轴折线图。

    横轴0表示仅包含内部变量和控制变量的基线模型。
    """

    sub = (
        results.loc[
            results["target"] == target
        ]
        .sort_values(
            "buffer_distance_m"
        )
        .copy()
    )


    color = get_target_color(
        target
    )

    marker = TARGET_MARKERS[
        target
    ]


    # -------------------------------------------------------------------------
    # 创建双y轴图
    # -------------------------------------------------------------------------

    fig, ax1 = plt.subplots(
        figsize=(8.6, 5.6),
        dpi=160,
    )

    ax2 = ax1.twinx()


    # -------------------------------------------------------------------------
    # 左轴：OOB R²
    # -------------------------------------------------------------------------

    line_r2 = ax1.plot(
        sub["buffer_distance_m"],
        sub["r2_oob"],
        color=color,
        linestyle="-",
        marker=marker,
        linewidth=2.2,
        markersize=6,
        label=f"{target} R²",
    )[0]


    # -------------------------------------------------------------------------
    # 右轴：OOB RMSE
    # -------------------------------------------------------------------------

    line_rmse = ax2.plot(
        sub["buffer_distance_m"],
        sub["rmse_oob"],
        color=color,
        linestyle="--",
        marker=marker,
        linewidth=2.2,
        markersize=6,
        markerfacecolor="white",
        label=f"{target} RMSE",
    )[0]


    # -------------------------------------------------------------------------
    # 突出显示0 m基线模型
    # -------------------------------------------------------------------------

    baseline = sub.loc[
        sub["buffer_distance_m"] == 0
    ]

    if not baseline.empty:

        ax1.scatter(
            baseline["buffer_distance_m"],
            baseline["r2_oob"],
            color=color,
            marker=marker,
            s=75,
            edgecolor="black",
            linewidth=0.8,
            zorder=10,
        )

        ax2.scatter(
            baseline["buffer_distance_m"],
            baseline["rmse_oob"],
            facecolor="white",
            edgecolor=color,
            marker=marker,
            s=75,
            linewidth=1.3,
            zorder=10,
        )


    # -------------------------------------------------------------------------
    # 坐标轴标题和图标题
    # -------------------------------------------------------------------------

    ax1.set_xlabel(
        "Buffer distance (m; 0 = baseline)",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax1.set_ylabel(
        "OOB R²",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax2.set_ylabel(
        "OOB RMSE",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax1.set_title(
        f"RF performance across buffer distances: {target}",
        fontsize=TITLE_SIZE_SINGLE,
    )


    # -------------------------------------------------------------------------
    # 统一设置左右坐标轴
    # -------------------------------------------------------------------------

    style_axes(
        ax1,
        ax2,
    )


    # -------------------------------------------------------------------------
    # 图例
    # -------------------------------------------------------------------------

    ax1.legend(
        [
            line_r2,
            line_rmse,
        ],
        [
            line_r2.get_label(),
            line_rmse.get_label(),
        ],
        loc="best",
        frameon=False,
        fontsize=LEGEND_SIZE_SINGLE,
    )


    # -------------------------------------------------------------------------
    # 保存图片
    # -------------------------------------------------------------------------

    fig.tight_layout()

    fig.savefig(
        OUTPUT_DIR
        / f"RF_{target}_R2_RMSE_dual_axis_with_baseline.png",
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        OUTPUT_DIR
        / f"RF_{target}_R2_RMSE_dual_axis_with_baseline.pdf",
        bbox_inches="tight",
    )

    plt.close(fig)


# =============================================================================
# 3. 六场景综合绘图
# =============================================================================

def plot_all_targets(
    results: pd.DataFrame,
) -> None:
    """
    将六个LST场景的R²与RMSE绘制在同一张双y轴图中。

    横轴0表示仅包含内部变量和控制变量的基线模型。
    """

    fig, ax1 = plt.subplots(
        figsize=(12.5, 7.2),
        dpi=160,
    )

    ax2 = ax1.twinx()


    legend_handles = []

    legend_labels = []


    # -------------------------------------------------------------------------
    # 循环绘制六个场景
    # -------------------------------------------------------------------------

    for target in TARGETS:

        sub = (
            results.loc[
                results["target"] == target
            ]
            .sort_values(
                "buffer_distance_m"
            )
            .copy()
        )


        color = get_target_color(
            target
        )

        marker = TARGET_MARKERS[
            target
        ]


        # 左轴：R²
        line_r2 = ax1.plot(
            sub["buffer_distance_m"],
            sub["r2_oob"],
            color=color,
            linestyle="-",
            marker=marker,
            linewidth=2.0,
            markersize=5.5,
            alpha=0.95,
            label=f"{target} R²",
        )[0]


        # 右轴：RMSE
        line_rmse = ax2.plot(
            sub["buffer_distance_m"],
            sub["rmse_oob"],
            color=color,
            linestyle="--",
            marker=marker,
            linewidth=2.0,
            markersize=5.5,
            markerfacecolor="white",
            alpha=0.95,
            label=f"{target} RMSE",
        )[0]


        # 突出显示0 m基线模型
        baseline = sub.loc[
            sub["buffer_distance_m"] == 0
        ]

        if not baseline.empty:

            ax1.scatter(
                baseline["buffer_distance_m"],
                baseline["r2_oob"],
                color=color,
                marker=marker,
                s=65,
                edgecolor="black",
                linewidth=0.7,
                zorder=10,
            )

            ax2.scatter(
                baseline["buffer_distance_m"],
                baseline["rmse_oob"],
                facecolor="white",
                edgecolor=color,
                marker=marker,
                s=65,
                linewidth=1.2,
                zorder=10,
            )


        legend_handles.extend(
            [
                line_r2,
                line_rmse,
            ]
        )

        legend_labels.extend(
            [
                line_r2.get_label(),
                line_rmse.get_label(),
            ]
        )


    # -------------------------------------------------------------------------
    # 坐标轴标题和图标题
    # -------------------------------------------------------------------------

    ax1.set_xlabel(
        "Buffer distance (m; 0 = baseline)",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax1.set_ylabel(
        "OOB R²",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax2.set_ylabel(
        "OOB RMSE",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax1.set_title(
        "Random forest performance across buffer distances",
        fontsize=TITLE_SIZE_ALL,
    )


    # -------------------------------------------------------------------------
    # 统一设置左轴、横轴和右轴
    # -------------------------------------------------------------------------

    style_axes(
        ax1,
        ax2,
    )


    # -------------------------------------------------------------------------
    # 图例
    # -------------------------------------------------------------------------

    ax1.legend(
        legend_handles,
        legend_labels,
        ncol=3,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.13),
        frameon=False,
        fontsize=LEGEND_SIZE_ALL,
    )


    # -------------------------------------------------------------------------
    # 保存图片
    # -------------------------------------------------------------------------

    fig.tight_layout(
        rect=[0, 0.08, 1, 1]
    )

    fig.savefig(
        OUTPUT_DIR
        / "RF_all_LST_R2_RMSE_dual_axis_with_baseline.png",
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        OUTPUT_DIR
        / "RF_all_LST_R2_RMSE_dual_axis_with_baseline.pdf",
        bbox_inches="tight",
    )

    plt.close(fig)


# =============================================================================
# 4. 主程序
# =============================================================================

def main() -> None:
    """
    主程序：
    读取数据、构建基线及缓冲区模型、输出结果并绘图。
    """

    print(
        f"正在读取数据：{DATA_PATH}"
    )


    # -------------------------------------------------------------------------
    # 读取数据
    # -------------------------------------------------------------------------

    df = pd.read_excel(
        DATA_PATH,
        sheet_name=0,
        engine="xlrd",
    )


    print(
        f"数据维度："
        f"{df.shape[0]}行 × "
        f"{df.shape[1]}列"
    )


    # -------------------------------------------------------------------------
    # 检查字段
    # -------------------------------------------------------------------------

    validate_columns(
        df
    )

    print(
        "字段检查通过。"
    )


    # -------------------------------------------------------------------------
    # 循环构建模型
    # -------------------------------------------------------------------------

    all_results = []

    total_models = (
        len(TARGETS)
        * len(MODEL_DISTANCES)
    )

    model_no = 0


    for target in TARGETS:

        for distance in MODEL_DISTANCES:

            model_no += 1

            model_type = get_model_type(
                distance
            )

            if distance == 0:
                model_description = (
                    "baseline（仅内部变量与控制变量）"
                )
            else:
                model_description = (
                    f"{distance} m缓冲区模型"
                )


            print(
                f"[{model_no:02d}/{total_models}] "
                f"建模："
                f"target={target}, "
                f"model={model_description}"
            )


            result = fit_one_rf(
                df=df,
                target=target,
                distance=distance,
            )


            all_results.append(
                result
            )


            print(
                f"    type={model_type}, "
                f"n={result['n_samples']}, "
                f"predictors={result['n_predictors']}, "
                f"OOB R²={result['r2_oob']:.4f}, "
                f"OOB RMSE={result['rmse_oob']:.4f}"
            )


    # -------------------------------------------------------------------------
    # 整理全部结果
    # -------------------------------------------------------------------------

    results = pd.DataFrame(
        all_results
    )

    results = (
        results
        .sort_values(
            [
                "target",
                "buffer_distance_m",
            ]
        )
        .reset_index(drop=True)
    )


    # -------------------------------------------------------------------------
    # 提取基线模型结果
    # -------------------------------------------------------------------------

    baseline_results = (
        results.loc[
            results["buffer_distance_m"] == 0
        ]
        .sort_values("target")
        .reset_index(drop=True)
    )


    # -------------------------------------------------------------------------
    # 仅保留30–300 m缓冲区模型
    # -------------------------------------------------------------------------

    buffer_results = (
        results.loc[
            results["buffer_distance_m"] > 0
        ]
        .copy()
    )


    # -------------------------------------------------------------------------
    # 在所有模型中按最高R²提取最优模型
    # -------------------------------------------------------------------------

    best_by_r2 = (
        results.loc[
            results
            .groupby("target")["r2_oob"]
            .idxmax()
        ]
        .sort_values("target")
        .reset_index(drop=True)
    )


    # -------------------------------------------------------------------------
    # 在所有模型中按最低RMSE提取最优模型
    # -------------------------------------------------------------------------

    best_by_rmse = (
        results.loc[
            results
            .groupby("target")["rmse_oob"]
            .idxmin()
        ]
        .sort_values("target")
        .reset_index(drop=True)
    )


    # -------------------------------------------------------------------------
    # 仅在30–300 m模型中按最高R²提取最佳缓冲距离
    # -------------------------------------------------------------------------

    best_buffer_by_r2 = (
        buffer_results.loc[
            buffer_results
            .groupby("target")["r2_oob"]
            .idxmax()
        ]
        .sort_values("target")
        .reset_index(drop=True)
    )


    # -------------------------------------------------------------------------
    # 仅在30–300 m模型中按最低RMSE提取最佳缓冲距离
    # -------------------------------------------------------------------------

    best_buffer_by_rmse = (
        buffer_results.loc[
            buffer_results
            .groupby("target")["rmse_oob"]
            .idxmin()
        ]
        .sort_values("target")
        .reset_index(drop=True)
    )


    # -------------------------------------------------------------------------
    # 计算最佳缓冲区模型相对于基线模型的性能变化
    # -------------------------------------------------------------------------

    baseline_compare = baseline_results[
        [
            "target",
            "r2_oob",
            "rmse_oob",
        ]
    ].rename(
        columns={
            "r2_oob": "baseline_r2_oob",
            "rmse_oob": "baseline_rmse_oob",
        }
    )

    best_buffer_comparison = (
        best_buffer_by_r2.merge(
            baseline_compare,
            on="target",
            how="left",
        )
    )

    best_buffer_comparison[
        "delta_r2_vs_baseline"
    ] = (
        best_buffer_comparison["r2_oob"]
        - best_buffer_comparison["baseline_r2_oob"]
    )

    best_buffer_comparison[
        "delta_rmse_vs_baseline"
    ] = (
        best_buffer_comparison["rmse_oob"]
        - best_buffer_comparison["baseline_rmse_oob"]
    )


    # -------------------------------------------------------------------------
    # 输出路径
    # -------------------------------------------------------------------------

    csv_path = (
        OUTPUT_DIR
        / "RF_buffer_distance_metrics_with_baseline.csv"
    )

    xlsx_path = (
        OUTPUT_DIR
        / "RF_buffer_distance_metrics_with_baseline.xlsx"
    )


    # -------------------------------------------------------------------------
    # 保存CSV
    # -------------------------------------------------------------------------

    results.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig",
    )


    # -------------------------------------------------------------------------
    # 保存Excel
    # -------------------------------------------------------------------------

    with pd.ExcelWriter(
        xlsx_path,
        engine="openpyxl",
    ) as writer:

        results.to_excel(
            writer,
            sheet_name="all_metrics",
            index=False,
        )

        baseline_results.to_excel(
            writer,
            sheet_name="baseline_models",
            index=False,
        )

        buffer_results.to_excel(
            writer,
            sheet_name="buffer_models",
            index=False,
        )

        best_by_r2.to_excel(
            writer,
            sheet_name="best_all_by_R2",
            index=False,
        )

        best_by_rmse.to_excel(
            writer,
            sheet_name="best_all_by_RMSE",
            index=False,
        )

        best_buffer_by_r2.to_excel(
            writer,
            sheet_name="best_buffer_by_R2",
            index=False,
        )

        best_buffer_by_rmse.to_excel(
            writer,
            sheet_name="best_buffer_by_RMSE",
            index=False,
        )

        best_buffer_comparison.to_excel(
            writer,
            sheet_name="buffer_vs_baseline",
            index=False,
        )


    # -------------------------------------------------------------------------
    # 绘制综合图
    # -------------------------------------------------------------------------

    plot_all_targets(
        results
    )


    # -------------------------------------------------------------------------
    # 绘制六张单场景图
    # -------------------------------------------------------------------------

    for target in TARGETS:

        plot_single_target(
            results,
            target,
        )


    # -------------------------------------------------------------------------
    # 输出运行结果
    # -------------------------------------------------------------------------

    print(
        "\n全部模型计算完成。"
    )

    print(
        f"结果表：{xlsx_path}"
    )

    print(
        "综合图："
        f"{OUTPUT_DIR / 'RF_all_LST_R2_RMSE_dual_axis_with_baseline.png'}"
    )


    print(
        "\n各场景基线模型性能："
    )

    print(
        baseline_results[
            [
                "target",
                "buffer_distance_m",
                "n_predictors",
                "r2_oob",
                "rmse_oob",
            ]
        ].to_string(
            index=False
        )
    )


    print(
        "\n各场景在所有模型中最高OOB R²对应的模型："
    )

    print(
        best_by_r2[
            [
                "target",
                "model_type",
                "buffer_distance_m",
                "r2_oob",
                "rmse_oob",
            ]
        ].to_string(
            index=False
        )
    )


    print(
        "\n各场景在30–300 m中最高OOB R²对应的缓冲距离："
    )

    print(
        best_buffer_by_r2[
            [
                "target",
                "buffer_distance_m",
                "r2_oob",
                "rmse_oob",
            ]
        ].to_string(
            index=False
        )
    )


    print(
        "\n最佳缓冲区模型相对于基线模型的性能变化："
    )

    print(
        best_buffer_comparison[
            [
                "target",
                "buffer_distance_m",
                "baseline_r2_oob",
                "r2_oob",
                "delta_r2_vs_baseline",
                "baseline_rmse_oob",
                "rmse_oob",
                "delta_rmse_vs_baseline",
            ]
        ].to_string(
            index=False
        )
    )


# =============================================================================
# 5. 运行主程序
# =============================================================================

if __name__ == "__main__":
    main()

正在读取数据：E:\excel\FILES\博士申请\第二篇论文\data\wuhuanshiliang\SH_attributes.xls
数据维度：1337行 × 166列
字段检查通过。
[01/66] 建模：target=lst0603, model=baseline（仅内部变量与控制变量）
    type=Baseline, n=1337, predictors=10, OOB R²=0.4129, OOB RMSE=1.1625
[02/66] 建模：target=lst0603, model=30 m缓冲区模型
    type=Buffer, n=1337, predictors=17, OOB R²=0.5054, OOB RMSE=1.0669
[03/66] 建模：target=lst0603, model=60 m缓冲区模型
    type=Buffer, n=1337, predictors=17, OOB R²=0.5302, OOB RMSE=1.0398
[04/66] 建模：target=lst0603, model=90 m缓冲区模型
    type=Buffer, n=1337, predictors=17, OOB R²=0.5415, OOB RMSE=1.0273
[05/66] 建模：target=lst0603, model=120 m缓冲区模型
    type=Buffer, n=1337, predictors=17, OOB R²=0.5336, OOB RMSE=1.0361
[06/66] 建模：target=lst0603, model=150 m缓冲区模型
    type=Buffer, n=1337, predictors=17, OOB R²=0.5255, OOB RMSE=1.0450
[07/66] 建模：target=lst0603, model=180 m缓冲区模型
    type=Buffer, n=1337, predictors=17, OOB R²=0.5252, OOB RMSE=1.0453
[08/66] 建模：target=lst0603, model=210 m缓冲区模型
    type=Buffer, n=1337, predictors=17, OOB R